In [2]:
import os
import gc
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from transformers import (
    GenerationConfig,
    MBartForConditionalGeneration,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer
)
import re
import torch
import pandas as pd
import unicodedata
from pathlib import Path
from sklearn.model_selection import train_test_split
from datasets import Dataset

In [3]:
TRAIN_SOURCE_PATH = Path('data/transcription_train.txt')
TRAIN_TARGET_PATH = Path('data/english_train.txt')
TEST_SOURCE_PATH = Path('data/transcription_validation.txt')
TEST_TARGET_PATH = Path('data/english_validation.txt')
CLEAN_TRAIN_OUTPUT = Path('data/train_cleaned.csv')
CLEAN_VAL_OUTPUT = Path('data/validation_cleaned.csv')
CLEAN_TEST_OUTPUT = Path('data/test_cleaned.csv')
TRAIN_SPLIT_RATIO = 0.9
RANDOM_STATE = 42
BLOCK_MIN_LINES = 2
BLOCK_MAX_LINES = 3
MAX_LENGTH_RATIO = 3.0
MIN_ID_SIGN_COUNT = 3
USE_BLOCKS = True
FAST_TEST_RUN = True
BLOCK_LENGTH = 256 
MAX_TRAIN_SAMPLES = 1200 if FAST_TEST_RUN else None
MAX_VAL_SAMPLES = 200 if FAST_TEST_RUN else None
MAX_TEST_SAMPLES = 200 if FAST_TEST_RUN else None
MAX_SOURCE_LENGTH = BLOCK_LENGTH
MAX_TARGET_LENGTH = BLOCK_LENGTH
NUM_TRAIN_EPOCHS = 1 if FAST_TEST_RUN else 5
MAX_TRAIN_STEPS = 200 if FAST_TEST_RUN else -1
GRID_SEARCH_MAX_STEPS = 80 if FAST_TEST_RUN else 200
TRAIN_EVAL_SAVE_STEPS = 50 if FAST_TEST_RUN else 500
TRAINING_OUTPUT_DIR = f'./mbart-akkadian-to-en-test-no-blocks-len-{BLOCK_LENGTH}' if not USE_BLOCKS else f'./mbart-akkadian-to-en-test-len-{BLOCK_LENGTH}'

In [4]:
# translation table for mapping normal digits to Unicode subscript digits
# translation tables define how to replace characters in a string using the str.translate() method 
SUBSCRIPT_DIGITS = str.maketrans('0123456789', '₀₁₂₃₄₅₆₇₈₉')

# Normalize determinatives between curly braces with expanded scholarly variations
def _normalize_determinative(match):
    content = match.group(1).strip().lower()
    
    # Remove internal @v or ~v flags (e.g., {lu2@v} → {lu2})
    content = re.sub(r'[@~]v', '', content)
    
    # Standardize male markers
    if content in ('1', 'm', 'disz'):
        content = 'm'
    # Standardize female markers
    elif content in ('mi2', 'f', 'munus'):
        content = 'f'
    # Standardize city markers
    elif content in ('iri', 'uru'):
        content = 'uru'
    
    return '{' + content + '}'

# Standardize all subscripts (u2, u_2, u₂) to a single Unicode subscript format (u₂)
def _normalize_subscripts(text):
    def replace(match):
        base = match.group(1)
        index = match.group(2).replace('_', '').translate(SUBSCRIPT_DIGITS)
        return base + index
    # Regex match letters followed by a number with optional underscore
    return re.sub(r'(?<!\w)([A-Za-zšṣṭḫĝŋŠṢṬḪ]+)_?([0-9]+)', replace, text)

# Normalize representations of gaps (..., x, [ ], [..]) in Akkadian texts 
# to standardized tokens <gap> and <big_gap>
def _normalize_akkadian_gaps(text):
    text = re.sub(r'\.{4,}', ' <big_gap> ', text)
    text = re.sub(r'\.{3}', ' <gap> ', text)
    text = re.sub(r'\[(?:\s*\.){0,3}\s*\]', ' <gap> ', text)
    text = re.sub(r'\b(?:x|X)\b', ' <gap> ', text)
    text = re.sub(r'(?<!\w)\[\s*\]\b', ' <gap> ', text)
    text = re.sub(r'(?<!\w)\[\.{1,3}\](?!\w)', ' <gap> ', text)
    text = re.sub(r'(?<!\w)\[\.\.\](?!\w)', ' <gap> ', text)
    return text

# Removes annotation artifacts such as line numbering, obverse/reverse tags, ...
def _strip_scholarly_noise(text):
    text = re.sub(r'(?m)^\s*\d+\.\s*', '', text)
    text = re.sub(r'(?m)^\s*[ivxlcdm]+\s+\d+\'?\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'(?m)^\s*\d+\s+\d+\'?\s*', '', text)
    text = re.sub(r'\{([^{}]+?)(?:[@~](?:v|obverse|reverse|obv|rev))\}', lambda match: '{' + match.group(1) + '}', text, flags=re.IGNORECASE)
    text = re.sub(r'([^\s{}]+?)(?:[@~](?:v|obverse|reverse|obv|rev))\b', r'\1', text, flags=re.IGNORECASE)
    text = re.sub(r'(?i)\s*@(?:obverse|reverse|obv|rev)\b', ' ', text)
    text = re.sub(r'(?<!\w)[#!?]+(?!\w)', '', text)
    text = re.sub(r'(?i)\bo\s*$', '', text)
    return text

# Normalize Akkadian text by standardizing subscripts, determinatives, gaps, and removing annotation artifacts
def normalize_akkadian(text):
    if pd.isna(text):
        return ''
    text = unicodedata.normalize('NFKC', str(text))
    text = _normalize_subscripts(text)
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'\{([^}]+)\}', _normalize_determinative, text)
    text = _strip_scholarly_noise(text)
    text = _normalize_akkadian_gaps(text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Normalize English text by removing parenthetical content, standardizing gaps, and collapsing whitespace
# Lighter clean up since source transliteration is noisier than target translation
def normalize_english(text):
    if pd.isna(text):
        return ''
    text = unicodedata.normalize('NFKC', str(text))
    text = re.sub(r'\([^)]*\)', ' ', text)
    text = re.sub(r'\.{4,}', ' <big_gap> ', text)
    text = re.sub(r'\.{3}', ' <gap> ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Count identifiable signs in the Akkadian text, ignoring gaps and non-sign tokens
# Estimation of readable source content after normalization
def count_identifiable_signs(text):
    if not text:
        return 0
    tokens = re.findall(r'<big_gap>|<gap>|\{[^}]+\}|[^\s]+', text)
    sign_count = 0
    for token in tokens:
        if token in {'<gap>', '<big_gap>'}:
            continue
        if token.startswith('{') and token.endswith('}'):
            sign_count += 1
            continue
        parts = [part for part in re.split(r'[-\s]+', token) if part and part not in {'#', '!', '?', 'o'}]
        sign_count += len(parts)
    return sign_count

# Filter out pairs where either source or target is empty, 
# source has too few identifiable signs, 
# or length ratio is too extreme
# This avoids training on pairs that are empty, too damaged, or strongly length-misaligned.
def is_meaningful_pair(source_text, target_text):
    if not source_text or not target_text:
        return False
    source_core = source_text.replace('<gap>', '').replace('<big_gap>', '').strip()
    if not source_core:
        return False
    if count_identifiable_signs(source_text) < MIN_ID_SIGN_COUNT:
        return False
    source_length = len(source_text.split())
    target_length = len(target_text.split())
    if source_length == 0 or target_length == 0:
        return False
    length_ratio = max(source_length / target_length, target_length / source_length)
    return length_ratio <= MAX_LENGTH_RATIO

# Determine block sizes for grouping lines based on total number of rows, ensuring blocks of 2-3 lines
def build_block_sizes(total_rows):
    if total_rows < BLOCK_MIN_LINES:
        return []
    if total_rows == 2:
        return [2]
    remainder = total_rows % BLOCK_MAX_LINES
    if remainder == 0:
        return [3] * (total_rows // 3)
    if remainder == 1:
        if total_rows < 4:
            return []
        return [3] * (total_rows // 3 - 1) + [2, 2]
    return [3] * (total_rows // 3) + [2]

# Sentence Reconstruction: Group consecutive lines into blocks of 2-3 lines to create more meaningful training pairs
# mBart benefits from training on longer sequences, and many sentences in the dataset are split across multiple lines.
# Akkadian often puts the verb at the end of the sentence, by grouping lines the model can better understand the relationships
def group_into_blocks(frame):
    if frame.empty:
        return pd.DataFrame(columns=['source_text', 'target_text'])
    block_sizes = build_block_sizes(len(frame))
    grouped_rows = []
    start_index = 0
    for block_size in block_sizes:
        block = frame.iloc[start_index:start_index + block_size]
        source_text = ' '.join(block['source_text'].tolist()).strip()
        target_text = ' '.join(block['target_text'].tolist()).strip()
        if is_meaningful_pair(source_text, target_text):
            grouped_rows.append({'source_text': source_text, 'target_text': target_text})
        start_index += block_size
    return pd.DataFrame(grouped_rows, columns=['source_text', 'target_text'])

def load_clean_parallel_dataset(source_path, target_path, save_path=None, use_blocks=True):
    cleaned_rows = []
    with open(source_path, encoding='utf-8') as source_file, open(target_path, encoding='utf-8') as target_file:
        for source_line, target_line in zip(source_file, target_file):
            source_text = normalize_akkadian(source_line.rstrip('\n'))
            target_text = normalize_english(target_line.rstrip('\n'))
            if is_meaningful_pair(source_text, target_text):
                cleaned_rows.append({'source_text': source_text, 'target_text': target_text})
    cleaned_frame = pd.DataFrame(cleaned_rows, columns=['source_text', 'target_text'])
    if use_blocks:
        final_frame = group_into_blocks(cleaned_frame)
    else:
        final_frame = cleaned_frame
    if save_path is not None:
        final_frame.to_csv(save_path, index=False)
    return final_frame

In [5]:
df_train_cleaned = load_clean_parallel_dataset(
    TRAIN_SOURCE_PATH,
    TRAIN_TARGET_PATH,
    use_blocks=USE_BLOCKS
)

df_train_cleaned = df_train_cleaned.rename(columns={
    'source_text': 'akkadian',
    'target_text': 'english'
})

df_train, df_val = train_test_split(
    df_train_cleaned,
    test_size=1 - TRAIN_SPLIT_RATIO,
    random_state=RANDOM_STATE,
    shuffle=True
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)

if MAX_TRAIN_SAMPLES is not None and len(df_train) > MAX_TRAIN_SAMPLES:
    df_train = df_train.sample(n=MAX_TRAIN_SAMPLES, random_state=RANDOM_STATE).reset_index(drop=True)

if MAX_VAL_SAMPLES is not None and len(df_val) > MAX_VAL_SAMPLES:
    df_val = df_val.sample(n=MAX_VAL_SAMPLES, random_state=RANDOM_STATE).reset_index(drop=True)

df_train.to_csv(CLEAN_TRAIN_OUTPUT, index=False)
df_val.to_csv(CLEAN_VAL_OUTPUT, index=False)

print('num samples train:', len(df_train))
print('num samples validation:', len(df_val))
print('saved cleaned train dataset to:', CLEAN_TRAIN_OUTPUT)
print('saved cleaned validation dataset to:', CLEAN_VAL_OUTPUT)
print(f'block grouping enabled: {USE_BLOCKS}')
print(f'current block length: {BLOCK_LENGTH}')

num samples train: 1200
num samples validation: 200
saved cleaned train dataset to: data\train_cleaned.csv
saved cleaned validation dataset to: data\validation_cleaned.csv
block grouping enabled: True
current block length: 256


In [6]:
print(df_train.shape)
df_train.head()

(1200, 2)


,akkadian,english
0,NAM.BUR₂.BI le-pu-u-šu tar-ba-ṣu la ka-aṣ-ru š...,Let them perform a namburbi ritual; the halo w...
1,<gap> i-na pa-an URU UR₅-tu₂ in-nam-mar-u₂ {m}...,<gap> Will they appear before that city? Mannu...
2,KUN₄.MEŠ {na₄}-DUR₂.MI.NA.BAN₃.DA {na₄}-GIŠ.NU...,I surrounded their lower courses with slabs of...
3,{m}-{d}-MAŠ—GIN—UN-MEŠ man-nu ša GIL-u-ni 05 M...,Ninurta-mukin-nishi. Whoever breaks the contra...
4,<gap> {giš}-MA₂ UR₅-tu₂ na-šu-u₂ <gap> a-na ša...,Will they plunder what is carried in that ship...


In [7]:
df_test_cleaned = load_clean_parallel_dataset(
    TEST_SOURCE_PATH,
    TEST_TARGET_PATH,
    save_path=CLEAN_TEST_OUTPUT,
    use_blocks=USE_BLOCKS
)

df_test = df_test_cleaned.rename(columns={
    'source_text': 'akkadian',
    'target_text': 'english'
})

if MAX_TEST_SAMPLES is not None and len(df_test) > MAX_TEST_SAMPLES:
    df_test = df_test.sample(n=MAX_TEST_SAMPLES, random_state=RANDOM_STATE).reset_index(drop=True)

print('num samples test cleaned:', len(df_test_cleaned))
print('saved cleaned test dataset to:', CLEAN_TEST_OUTPUT)

num samples test cleaned: 772
saved cleaned test dataset to: data\test_cleaned.csv


In [8]:
print(df_test.shape)
df_test.head()

(200, 2)


,akkadian,english
0,at-tu-nu ne₂-e-hu dul-la-šu₂-nu ep-pu-šu₂ TA Š...,They are peaceful and do their work. I have br...
1,{udu}-SISKUR ina IGI {d}-5.1.1 ina IGI ma-a.a-...,You perform a sheep offering before the Igigi ...
2,BE ŠA₃.NIGIN 14 ŠA₃-bi UDU.NITA₂ ša₂-lim EGIR-...,The coils of the colon are 14 in number. The h...
3,UGU DINGIR-ti-ka GAL-ti {d}-UTU EN GAL-u₂ lil-...,"May go to your great divinity, O Šamaš, great ..."
4,IGI {m}- <gap> DINGIR-a.a {lu₂}-GAL—ki-ṣir {lu...,"Witness <gap> -ila'i, cohort commander. In my ..."


In [9]:
# Check for any missing values
print(f"\n🔍 Missing Values Check:")
print(f"Train akkadian missing: {df_train['akkadian'].isnull().sum()}")
print(f"Train english missing: {df_train['english'].isnull().sum()}")
print(f"Validation akkadian missing: {df_val['akkadian'].isnull().sum()}")
print(f"Validation english missing: {df_val['english'].isnull().sum()}")
print(f"Test akkadian missing: {df_test['akkadian'].isnull().sum()}")
print(f"Test english missing: {df_test['english'].isnull().sum()}")


🔍 Missing Values Check:
Train akkadian missing: 0
Train english missing: 0
Validation akkadian missing: 0
Validation english missing: 0
Test akkadian missing: 0
Test english missing: 0


In [10]:
print(f"📊 Dataset Sizes:")
print(f"Training data shape: {df_train.shape}")
print(f"Validation data shape: {df_val.shape}")
print(f"Test data shape: {df_test.shape}")
print(f"Total examples: {len(df_train) + len(df_val) + len(df_test)}")

📊 Dataset Sizes:
Training data shape: (1200, 2)
Validation data shape: (200, 2)
Test data shape: (200, 2)
Total examples: 1600


In [11]:
# Check for GPU availability and set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️ Using device: {device}")

# Display GPU information if available
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

🖥️ Using device: cuda
GPU Name: NVIDIA GeForce RTX 4060 Laptop GPU
GPU Memory: 8.6 GB


In [12]:
# Convert pandas DataFrames to HuggingFace Dataset objects
# This format is required for the HuggingFace Trainer API
train_dataset = Dataset.from_pandas(df_train, split='train')
val_dataset = Dataset.from_pandas(df_val, split='validation')
test_dataset = Dataset.from_pandas(df_test, split='test')

print(f"✅ Created HuggingFace Datasets:")
print(f"Train dataset: {train_dataset}")
print(f"Validation dataset: {val_dataset}")
print(f"Test dataset: {test_dataset}")

✅ Created HuggingFace Datasets:
Train dataset: Dataset({
    features: ['akkadian', 'english'],
    num_rows: 1200
})
Validation dataset: Dataset({
    features: ['akkadian', 'english'],
    num_rows: 200
})
Test dataset: Dataset({
    features: ['akkadian', 'english'],
    num_rows: 200
})


In [13]:
# Load pre-trained mBART model and tokenizer with many-to-many configuration
from transformers import MBart50TokenizerFast

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model_name = 'facebook/mbart-large-50-many-to-many-mmt'


def load_model_and_tokenizer():
    model = MBartForConditionalGeneration.from_pretrained(model_name)
    tokenizer = MBart50TokenizerFast.from_pretrained(model_name)

    # Set source and target languages: use Arabic (ar_AR) as proxy for Akkadian
    tokenizer.src_lang = "ar_AR"
    tokenizer.tgt_lang = "en_XX"

    # Add custom gap tokens as special tokens so they are never split into subwords
    tokenizer.add_special_tokens({'additional_special_tokens': ["<gap>", "<big_gap>"]})

    # Resize model embeddings immediately after extending the tokenizer
    model.resize_token_embeddings(len(tokenizer))

    # Reduce memory use during training
    model.gradient_checkpointing_enable()
    model.config.use_cache = False

    # mBART generation needs explicit decoder start / forced BOS language tokens
    if hasattr(tokenizer, "lang_code_to_id") and "en_XX" in tokenizer.lang_code_to_id:
        en_lang_id = tokenizer.lang_code_to_id["en_XX"]
        model.config.decoder_start_token_id = en_lang_id
        model.config.forced_bos_token_id = en_lang_id
        model.generation_config.decoder_start_token_id = en_lang_id
        model.generation_config.forced_bos_token_id = en_lang_id

    return model, tokenizer


model, tokenizer = load_model_and_tokenizer()

print(f"Model parameters: {model.num_parameters():,}")
print(f"Tokenizer vocabulary size: {len(tokenizer)}")
print(f"Tokenizer type: {type(tokenizer).__name__}")
print(f"Source language: {tokenizer.src_lang}, Target language: {tokenizer.tgt_lang}")
print(f"Decoder start token id: {model.config.decoder_start_token_id}")
print(f"Forced BOS token id: {model.config.forced_bos_token_id}")
print("✅ Custom special tokens added: <gap>, <big_gap>")

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Model parameters: 610,881,536
Tokenizer vocabulary size: 250056
Tokenizer type: MBart50TokenizerFast
Source language: ar_AR, Target language: en_XX
Decoder start token id: 250004
Forced BOS token id: 250004
✅ Custom special tokens added: <gap>, <big_gap>


In [14]:
def preprocess_function(examples):
    # Keep tokenization lightweight; dynamic padding is handled by the collator.
    # Use a shorter max length in fast-test mode to keep runs short.
    inputs = tokenizer(
        examples["akkadian"],
        truncation=True,
        max_length=MAX_SOURCE_LENGTH,
        return_tensors=None
    )

    labels = tokenizer(
        examples["english"],
        truncation=True,
        max_length=MAX_TARGET_LENGTH,
        return_tensors=None
    )

    inputs["labels"] = labels["input_ids"]
    return inputs

# Tokenize training, validation, and test datasets
print("🔄 Tokenizing training dataset...")
tokenized_train_dataset = train_dataset.map(preprocess_function, batched=True, desc="Processing train")

print("🔄 Tokenizing validation dataset...")
tokenized_val_dataset = val_dataset.map(preprocess_function, batched=True, desc="Processing validation")

print("🔄 Tokenizing test dataset...")
tokenized_test_dataset = test_dataset.map(preprocess_function, batched=True, desc="Processing test")

print("✅ Tokenization complete")

🔄 Tokenizing training dataset...


Processing train:   0%|          | 0/1200 [00:00<?, ? examples/s]

🔄 Tokenizing validation dataset...


Processing validation:   0%|          | 0/200 [00:00<?, ? examples/s]

🔄 Tokenizing test dataset...


Processing test:   0%|          | 0/200 [00:00<?, ? examples/s]

✅ Tokenization complete


In [15]:
# Set up generation config with optimized parameters for Akkadian-to-English translation

generation_config = GenerationConfig(
    max_new_tokens=384,
    num_beams=2,
    repetition_penalty=2.0,
    no_repeat_ngram_size=3,
    early_stopping=True,
)


def build_training_args(output_dir, config):
    return Seq2SeqTrainingArguments(
        output_dir=str(output_dir),
        per_device_train_batch_size=config.get("per_device_train_batch_size", 2),
        per_device_eval_batch_size=config.get("per_device_eval_batch_size", 2),
        gradient_accumulation_steps=config.get("gradient_accumulation_steps", 4),
        learning_rate=config["learning_rate"],
        weight_decay=config["weight_decay"],
        warmup_ratio=config["warmup_ratio"],
        num_train_epochs=NUM_TRAIN_EPOCHS,
        max_steps=config.get("max_steps", MAX_TRAIN_STEPS),
        save_total_limit=1,
        # Faster iteration: evaluate by loss only during training
        predict_with_generate=False,
        generation_config=generation_config,
        fp16=torch.cuda.is_available(),
        eval_strategy="steps",
        eval_steps=TRAIN_EVAL_SAVE_STEPS,
        save_strategy="steps",
        save_steps=TRAIN_EVAL_SAVE_STEPS,
        logging_steps=100,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        optim="adafactor",
        report_to="none",
    )


grid_search_configs = [
    {
        "name": "baseline",
        "learning_rate": 2e-5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.03,
        "gradient_accumulation_steps": 4,
    },
    {
        "name": "higher_lr",
        "learning_rate": 3e-5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.03,
        "gradient_accumulation_steps": 4,
    },
    {
        "name": "lower_lr_more_warmup",
        "learning_rate": 1e-5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.06,
        "gradient_accumulation_steps": 4,
    },
    {
        "name": "no_decay_more_warmup",
        "learning_rate": 1e-5,
        "weight_decay": 0.0,
        "warmup_ratio": 0.03,
        "gradient_accumulation_steps": 4,
    },
]


def run_grid_search_trial(config):
    trial_name = config["name"]
    trial_output_dir = Path(TRAINING_OUTPUT_DIR) / "grid_search" / trial_name
    trial_model, trial_tokenizer = load_model_and_tokenizer()
    trial_args = build_training_args(trial_output_dir, {**config, "max_steps": GRID_SEARCH_MAX_STEPS})
    trial_collator = DataCollatorForSeq2Seq(
        tokenizer=trial_tokenizer,
        model=trial_model,
        label_pad_token_id=-100,
        pad_to_multiple_of=8 if torch.cuda.is_available() else None,
    )
    trial_trainer = Seq2SeqTrainer(
        model=trial_model,
        args=trial_args,
        train_dataset=tokenized_train_dataset,
        eval_dataset=tokenized_val_dataset,
        data_collator=trial_collator,
    )

    print(f"\n🔎 Trial: {trial_name}")
    print(
        f"   lr={config['learning_rate']}, weight_decay={config['weight_decay']}, "
        f"warmup_ratio={config['warmup_ratio']}, grad_accum={config['gradient_accumulation_steps']}"
    )

    trial_trainer.train()
    trial_metrics = trial_trainer.evaluate()
    eval_loss = trial_metrics.get("eval_loss", float("inf"))

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    del trial_trainer, trial_collator, trial_model, trial_tokenizer
    gc.collect()

    return {
        **config,
        "eval_loss": eval_loss,
        "output_dir": str(trial_output_dir),
    }


print("\n🚦 Starting compact grid search...")
Path(TRAINING_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
search_results = [run_grid_search_trial(config) for config in grid_search_configs]
results_df = pd.DataFrame(search_results).sort_values("eval_loss").reset_index(drop=True)
results_df.to_csv(Path(TRAINING_OUTPUT_DIR) / "grid_search_results.csv", index=False)

print("\n📈 Grid search results (sorted by validation loss):")
print(results_df[["name", "learning_rate", "weight_decay", "warmup_ratio", "gradient_accumulation_steps", "eval_loss"]].to_string(index=False))

best_config = results_df.iloc[0].to_dict()
print(f"\n✅ Best config: {best_config['name']} with eval_loss={best_config['eval_loss']:.4f}")

# Rebuild the global training objects using the winning configuration so the next cells train the best model.
training_args = build_training_args(Path(TRAINING_OUTPUT_DIR), {**best_config, "max_steps": MAX_TRAIN_STEPS})
model, tokenizer = load_model_and_tokenizer()
print(f"Using best config for final training: {best_config['name']}")


🚦 Starting compact grid search...


c:\Users\woutv\AppData\Local\Programs\Python\Python313\Lib\site-packages\accelerate\accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
c:\Users\woutv\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\nn\modules\module.py:1329: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  return t.to(



🔎 Trial: no_decay
   lr=1e-05, weight_decay=0.0, warmup_ratio=0.03, grad_accum=4


Step,Training Loss,Validation Loss
50,No log,4.066258


c:\Users\woutv\AppData\Local\Programs\Python\Python313\Lib\site-packages\transformers\modeling_utils.py:2758: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200, 'early_stopping': True, 'num_beams': 5, 'forced_bos_token_id': 250004}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].



📈 Grid search results (sorted by validation loss):
    name  learning_rate  weight_decay  warmup_ratio  gradient_accumulation_steps  eval_loss
no_decay        0.00001           0.0          0.03                            4   4.066258

✅ Best config: no_decay with eval_loss=4.0663
Using best config for final training: no_decay


First try:
📈 Grid search results (sorted by validation loss):
                name  learning_rate  weight_decay  warmup_ratio  gradient_accumulation_steps  eval_loss
           higher_lr        0.00003          0.01          0.03                            4   3.533545
            baseline        0.00002          0.01          0.03                            4   3.707581
no_decay_more_warmup        0.00002          0.00          0.06                            4   3.710583
lower_lr_more_warmup        0.00001          0.01          0.06                            4   4.086250

✅ Best config: higher_lr with eval_loss=3.5335
Using best config for final training: higher_lr